## This notebook aims to test loading data directly from the server

In [36]:
import requests
import rasterio
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import PurePosixPath
from rasterio.io import MemoryFile

In [37]:
BASE_URL = "http://206.12.92.143/data/dashboard/"

response = requests.get(BASE_URL, timeout=10)
response.raise_for_status()

print(f"Status: {response.status_code}")

# Parse directory listing HTML
soup = BeautifulSoup(response.text, "html.parser")

files = []

for link in soup.find_all("a"):
    href = link.get("href")

    full_url = urljoin(BASE_URL, href)
    files.append(full_url)

print("Files found:")
for f in files:
    print(f)

Status: 200
Files found:
http://206.12.92.143/data/dashboard/?C=N;O=D
http://206.12.92.143/data/dashboard/?C=M;O=A
http://206.12.92.143/data/dashboard/?C=S;O=A
http://206.12.92.143/data/dashboard/?C=D;O=A
http://206.12.92.143/data/
http://206.12.92.143/data/dashboard/ALFL/
http://206.12.92.143/data/dashboard/AMCR/
http://206.12.92.143/data/dashboard/AMGO/
http://206.12.92.143/data/dashboard/AMPI/
http://206.12.92.143/data/dashboard/AMRE/
http://206.12.92.143/data/dashboard/AMRO/
http://206.12.92.143/data/dashboard/ATSP/
http://206.12.92.143/data/dashboard/ATTW/
http://206.12.92.143/data/dashboard/BANS/
http://206.12.92.143/data/dashboard/BAOR/
http://206.12.92.143/data/dashboard/BARS/
http://206.12.92.143/data/dashboard/BAWW/
http://206.12.92.143/data/dashboard/BBCU/
http://206.12.92.143/data/dashboard/BBMA/
http://206.12.92.143/data/dashboard/BBWA/
http://206.12.92.143/data/dashboard/BBWO/
http://206.12.92.143/data/dashboard/BCCH/
http://206.12.92.143/data/dashboard/BEKI/
http://206.1

In [38]:
def list_remote_files(base_url, extension=None):
    """
    List files from an Apache directory index.
    """

    response = requests.get(base_url, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    files = []

    for link in soup.find_all("a"):
        href = link.get("href")

        if href in [None, "../"]:
            continue

        full_url = urljoin(base_url, href)

        if extension:
            if PurePosixPath(href).suffix.lower() != extension.lower():
                continue

        files.append(full_url)

    return files


# Example: list all tif files
tif_files = list_remote_files(BASE_URL, extension=".tif")

print("Remote TIFF files:")
for f in tif_files:
    print(f)


def open_remote_tif(url):
    """
    Load a remote TIFF directly into rasterio memory.
    """

    response = requests.get(url)
    response.raise_for_status()

    with MemoryFile(response.content) as memfile:
        with memfile.open() as dataset:
            array = dataset.read(1)
            profile = dataset.profile

    return array, profile


# Example: open first tif
if tif_files:
    arr, profile = open_remote_tif(tif_files[0])

    print("\nShape:", arr.shape)
    print("CRS:", profile["crs"])

Remote TIFF files:


In [39]:
from localtileserver import TileClient, get_leaflet_tile_layer
from ipyleaflet import Map, LayersControl
import rasterio
import requests
import tempfile

# Remote COG URL
url = "http://206.12.92.143/data/dashboard/ALFL/Canada/ALFL_Canada_1990.tif"


# ---------------------------------------------------------
# Download remote TIFF to temporary local file
# ---------------------------------------------------------
response = requests.get(url, stream=True, timeout=120)
response.raise_for_status()

tmp = tempfile.NamedTemporaryFile(suffix=".tif", delete=False)

for chunk in response.iter_content(chunk_size=8192):
    tmp.write(chunk)

tmp.close()

local_path = tmp.name

print("Temporary file:", local_path)


# ---------------------------------------------------------
# Create localtileserver client
# ---------------------------------------------------------
client = TileClient(local_path)

layer = get_leaflet_tile_layer(
    client,
    opacity=0.8,
)


# ---------------------------------------------------------
# Get map center from raster bounds
# ---------------------------------------------------------
with rasterio.open(local_path) as src:
    bounds = src.bounds

center_lat = (bounds.bottom + bounds.top) / 2
center_lon = (bounds.left + bounds.right) / 2


# ---------------------------------------------------------
# Create ipyleaflet map
# ---------------------------------------------------------
m = Map(
    center=(center_lat, center_lon),
    zoom=4,
)

m.add(layer)
m.add(LayersControl())

m

Temporary file: /var/folders/gw/x48yk1kn6cgd9wzx3mlttb0h0000gn/T/tmphy6o22ca.tif


Map(center=[55.516878464556584, -91.2933004053187], controls=(ZoomControl(options=['position', 'zoom_in_text',…

In [50]:
parent_folder = 'dashboard/'
def list_directory(url: str) -> list[str]:
    """
    Parse Apache directory listing and return entries.
    """

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    entries = []

    for link in soup.find_all("a"):
        href = link.get("href")

        if href not in [None, "../", '/data/', '?C=N;O=D', '?C=M;O=A', '?C=S;O=A', '?C=D;O=A', f'/data/{parent_folder}']:
            entries.append(href.rstrip("/"))
            continue

    return entries

list_directory(BASE_URL)

['ALFL',
 'AMCR',
 'AMGO',
 'AMPI',
 'AMRE',
 'AMRO',
 'ATSP',
 'ATTW',
 'BANS',
 'BAOR',
 'BARS',
 'BAWW',
 'BBCU',
 'BBMA',
 'BBWA',
 'BBWO',
 'BCCH',
 'BEKI',
 'BGGN',
 'BHCO',
 'BHVI',
 'BLBW',
 'BLJA',
 'BLPW',
 'BOBO',
 'BOCH',
 'BOWA',
 'BRBL',
 'BRCR',
 'BRTH',
 'BTBW',
 'BTNW',
 'BWWA',
 'CAJA',
 'CAWA',
 'CCSP',
 'CEDW',
 'CHSP',
 'CLSW',
 'CMWA',
 'COGR',
 'CONW',
 'CORA',
 'COYE',
 'CSWA',
 'DEJU',
 'DOWO',
 'DUFL',
 'DUNL',
 'EABL',
 'EAKI',
 'EAPH',
 'EATO',
 'EAWP',
 'EUST',
 'EVGR',
 'FISP',
 'FOSP',
 'GCFL',
 'GCKI',
 'GCSP',
 'GCTH',
 'GRCA',
 'GRSP',
 'GRYE',
 'GWWA',
 'HAFL',
 'HAWO',
 'HETH',
 'HOLA',
 'HOSP',
 'HOWR',
 'INBU',
 'KILL',
 'LALO',
 'LCSP',
 'LEFL',
 'LEYE',
 'LISP',
 'MAWA',
 'MAWR',
 'MOBL',
 'MODO',
 'MOWA',
 'NAWA',
 'NESP',
 'NOCA',
 'NOFL',
 'NOPA',
 'NOWA',
 'OCWA',
 'OSFL',
 'OVEN',
 'PAWA',
 'PHVI',
 'PIGR',
 'PISI',
 'PIWA',
 'PIWO',
 'PUFI',
 'RBGR',
 'RBNU',
 'RBWO',
 'RCKI',
 'RECR',
 'REVI',
 'RHWO',
 'ROPI',
 'ROSA',
 'RTHU',
 'RUBL',
 

In [54]:
def available_regions(species_id: str) -> list[str]:
    """
    List regions for a species.
    """

    species_url = urljoin(BASE_URL, f"{species_id}/")

    try:
        entries = list_directory(species_url)

        return sorted(entries)

    except Exception:
        return []
available_regions("CAWA")

['Canada', 'Lower48']

In [ ]:
def available_years(species_id: str, region: str) -> list[int]:
    """
    Discover available years by parsing remote TIFF filenames.
    """

    region_url = urljoin(
        BASE_URL,
        f"{species_id}/{region}/"
    )

    try:
        entries = list_directory(region_url)

    except Exception:
        return []

    years = []

    prefix = f"{species_id}_{region}_"

    for filename in entries:
        if not filename.endswith(".tif"):
            continue

        stem = filename.removesuffix(".tif")

        if not stem.startswith(prefix):
            continue

        try:
            year = int(stem.removeprefix(prefix))
            years.append(year)

        except ValueError:
            continue

    return sorted(years)

available_years("CAWA", "Canada")

['/data/dashboard/CAWA', 'CAWA_Canada_1990.tif', 'CAWA_Canada_1995.tif', 'CAWA_Canada_2000.tif', 'CAWA_Canada_2005.tif', 'CAWA_Canada_2010.tif', 'CAWA_Canada_2015.tif', 'CAWA_Canada_2020.tif']


[1990, 1995, 2000, 2005, 2010, 2015, 2020]